# 🇺🇾 Análisis de Personas Políticamente Expuestas (PEP) — Uruguay

**Proyecto de Análisis de Datos para Prevención de Lavado de Activos**  
**Marco normativo:** Ley N° 19.574 | **Fuente:** SENACLAFT  

---

## 📋 Contenido

| Sección | Descripción |
|---------|-------------|
| 1. Setup | Instalación e importación de librerías |
| 2. Carga de Datos | Dataset oficial SENACLAFT (857 PEPs) |
| 3. Feature Engineering | Categorización de riesgo, organismo y edad |
| 4. Visualizaciones | 5 gráficos con estilo oscuro profesional |
| 5. Conclusiones | Hallazgos clave y tabla resumen |

> ⚠️ Los datos son públicos y oficiales. No contienen información sensible adicional.

## 1. ⚙️ Setup e Instalación

Instalamos las librerías necesarias (compatibles con Google Colab).

In [ ]:
# Instalación de dependencias (descomentar en Google Colab)
# !pip install pandas numpy matplotlib seaborn -q

import os
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

print(f"Python  : {sys.version.split()[0]}")
print(f"Pandas  : {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

### 1.1 Ruta de datos

El notebook detecta automáticamente si se ejecuta en Google Colab o localmente.

In [ ]:
# Detección de entorno y configuración de rutas
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Clonar el repositorio si se ejecuta en Google Colab
    if not os.path.exists("Analisis-PEP-Uruguay"):
        os.system("git clone https://github.com/federicoramos67/Analisis-PEP-Uruguay.git")
    os.chdir("Analisis-PEP-Uruguay")
    DATA_PATH = "data/2.lista-actualizada-de-pep.csv"
    sys.path.insert(0, "src")
else:
    # Ejecutar localmente desde la raíz del proyecto
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    project_root = os.path.dirname(notebook_dir)
    os.chdir(project_root)
    DATA_PATH = "data/2.lista-actualizada-de-pep.csv"
    sys.path.insert(0, "src")

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Directorio de trabajo: {os.getcwd()}")

### 1.2 Paleta de colores y estilo

Definimos el estilo visual consistente para todos los gráficos.

In [ ]:
# === PALETA DE COLORES Y ESTILO GLOBAL ===

# Colores del proyecto
C_BG      = "#0f0f17"   # Fondo oscuro
C_BG2     = "#1a1a27"   # Fondo secundario (paneles)
C_RED     = "#e63946"   # Rojo principal (alto riesgo)
C_TEAL    = "#2a9d8f"   # Verde-teal (medio riesgo)
C_GOLD    = "#f4a261"   # Dorado (medio-alto)
C_PURPLE  = "#7b5ea7"   # Púrpura (otros)
C_TEXT    = "#e8e8f0"   # Texto principal
C_MUTED   = "#7a7a9a"   # Texto secundario

# Paleta para niveles de riesgo (ordenados)
RISK_COLORS = {
    "Alto - Ejecutivo/Legislativo":   C_RED,
    "Alto - Poder Judicial":           "#c1121f",
    "Alto - Dirección General":        "#ff6b6b",
    "Medio-Alto - Diplomático/Policial": C_GOLD,
    "Medio - Dirección/Gerencia":      C_TEAL,
    "Medio - Representación Local":    "#52b788",
    "Otros":                           C_PURPLE,
}

# Configuración global de matplotlib
plt.rcParams.update({
    "figure.facecolor":  C_BG,
    "axes.facecolor":    C_BG2,
    "axes.edgecolor":    C_MUTED,
    "axes.labelcolor":   C_TEXT,
    "text.color":        C_TEXT,
    "xtick.color":       C_MUTED,
    "ytick.color":       C_MUTED,
    "grid.color":        "#2a2a3e",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "font.family":       "sans-serif",
    "font.size":         11,
    "axes.titlesize":    14,
    "axes.titleweight":  "bold",
    "axes.titlepad":     14,
})

print("✅ Estilo configurado correctamente.")

## 2. 📂 Carga y Exploración de Datos

El dataset es la **lista oficial de PEPs** publicada por SENACLAFT en el Catálogo de Datos Abiertos del Gobierno de Uruguay.

In [ ]:
# Carga del dataset
df_raw = pd.read_csv(DATA_PATH)

print("=" * 50)
print(f"  Registros totales : {len(df_raw):,}")
print(f"  Columnas          : {df_raw.columns.tolist()}")
print(f"  Duplicados        : {df_raw.duplicated().sum()}")
print(f"  Valores nulos     : {df_raw.isnull().sum().sum()}")
print("=" * 50)

df_raw.head(5)

## 3. 🔧 Feature Engineering

Creamos tres columnas adicionales:

| Columna | Descripción |
|---------|-------------|
|  | Categoría de riesgo según cargo (AML/KYC) |
|  | Clasificación del organismo por naturaleza institucional |
|  | Estimación de rango etario según numeración de CI |


In [ ]:
from analisis_pep import categorizar_cargo_v2, categorizar_organismo, estimar_edad

df = df_raw.copy()
df["NIVEL_RIESGO"]  = df["CARGO"].apply(categorizar_cargo_v2)
df["TIPO_ORGANISMO"] = df["ORGANISMO"].apply(categorizar_organismo)
df["RANGO_EDAD"]     = df["CI"].apply(estimar_edad)

# Clasificación simplificada de riesgo
df["RIESGO_SIMPLE"] = df["NIVEL_RIESGO"].apply(
    lambda x: "Alto" if "Alto" in x else ("Medio-Alto" if "Medio-Alto" in x else
              ("Medio" if "Medio" in x else "Otros"))
)

print(f"✅ Dataset enriquecido: {df.shape}")
print()
print("Distribución por nivel de riesgo:")
print(df["NIVEL_RIESGO"].value_counts())

## 4. 📊 Visualizaciones

Generamos **5 visualizaciones profesionales** con estilo oscuro consistente.

---


### 4.1 Distribución por Nivel de Riesgo

Barras horizontales ordenadas de mayor a menor, con coloreado semántico por nivel de riesgo.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

risk_counts = df["NIVEL_RIESGO"].value_counts().sort_values()
labels = risk_counts.index.tolist()
values = risk_counts.values
colors = [RISK_COLORS.get(l, C_PURPLE) for l in labels]

bars = ax.barh(labels, values, color=colors, height=0.65, zorder=3)

# Etiquetas de valor
for bar, val in zip(bars, values):
    pct = val / len(df) * 100
    ax.text(
        bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
        f"{val}  ({pct:.1f}%)",
        va="center", ha="left", fontsize=10, color=C_TEXT
    )

ax.set_xlim(0, values.max() * 1.25)
ax.set_xlabel("Cantidad de PEPs", labelpad=10)
ax.set_title("Distribución de los 857 PEPs por Nivel de Riesgo")
ax.grid(axis="x", zorder=0)
ax.tick_params(axis="y", labelsize=10)

# Línea total
ax.axvline(x=229, color=C_MUTED, linestyle=":", alpha=0.4)

# Etiqueta resumen alto riesgo
alto_total = df["RIESGO_SIMPLE"].eq("Alto").sum()
ax.text(
    0.98, 0.04,
    f"Total Alto Riesgo: {alto_total} ({alto_total/len(df)*100:.1f}%)",
    transform=ax.transAxes, ha="right", va="bottom",
    fontsize=10, color=C_RED,
    bbox=dict(boxstyle="round,pad=0.4", facecolor=C_BG, edgecolor=C_RED, alpha=0.8)
)

fig.tight_layout()
plt.savefig("notebooks/viz_01_riesgo_distribucion.png", dpi=150,
            bbox_inches="tight", facecolor=C_BG)
plt.show()
print("✅ Gráfico guardado.")

### 4.2 Top 15 Organismos con Más PEPs

Organismos con mayor representación en el registro, coloreados por tipo institucional.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

top_org = df["ORGANISMO"].value_counts().head(15)

# Color por tipo de organismo
ORG_TYPE_COLORS = {
    "Poder Legislativo":          "#4cc9f0",
    "Ministerios":                 C_RED,
    "Poder Judicial":              "#f77f00",
    "Otros Organismos":            C_PURPLE,
    "Gobiernos Departamentales":   "#52b788",
    "Instituciones Financieras":   C_GOLD,
    "Empresas Públicas":           "#90e0ef",
    "Educación":                   "#b5838d",
}

bar_colors = [
    ORG_TYPE_COLORS.get(df[df["ORGANISMO"] == org]["TIPO_ORGANISMO"].iloc[0], C_PURPLE)
    for org in top_org.index
]

# Abreviar nombres largos
def abrev(name, maxlen=38):
    return name if len(name) <= maxlen else name[:maxlen] + "..."

short_labels = [abrev(n) for n in top_org.index]

bars = ax.bar(range(len(top_org)), top_org.values, color=bar_colors, width=0.7, zorder=3)

# Etiquetas de valor encima
for i, (bar, val) in enumerate(zip(bars, top_org.values)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(val), ha="center", va="bottom", fontsize=9, color=C_TEXT)

ax.set_xticks(range(len(top_org)))
ax.set_xticklabels(short_labels, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Cantidad de PEPs", labelpad=10)
ax.set_title("Top 15 Organismos con Mayor Cantidad de PEPs Registrados")
ax.grid(axis="y", zorder=0)

# Leyenda por tipo
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in ORG_TYPE_COLORS.items()
                  if k in df["TIPO_ORGANISMO"].unique()]
ax.legend(handles=legend_patches, loc="upper right", fontsize=8,
          facecolor=C_BG, edgecolor=C_MUTED, framealpha=0.8)

fig.tight_layout()
plt.savefig("notebooks/viz_02_top_organismos.png", dpi=150,
            bbox_inches="tight", facecolor=C_BG)
plt.show()
print("✅ Gráfico guardado.")

### 4.3 Distribución por Tipo de Organismo

Gráfico de dona mostrando la composición institucional de los PEPs registrados.

In [ ]:
fig, (ax_donut, ax_table) = plt.subplots(1, 2, figsize=(14, 7),
                                          gridspec_kw={"width_ratios": [1.2, 1]})

org_counts = df["TIPO_ORGANISMO"].value_counts()

DONUT_COLORS = [
    "#4cc9f0", C_RED, "#f77f00", C_GOLD, "#52b788", C_PURPLE, "#90e0ef", "#b5838d"
]

wedges, texts, autotexts = ax_donut.pie(
    org_counts.values,
    labels=None,
    colors=DONUT_COLORS[:len(org_counts)],
    autopct=lambda pct: f"{pct:.1f}%" if pct > 4 else "",
    pctdistance=0.78,
    startangle=90,
    wedgeprops={"width": 0.55, "edgecolor": C_BG, "linewidth": 2},
)

for t in autotexts:
    t.set_fontsize(9)
    t.set_color(C_BG)
    t.set_fontweight("bold")

# Centro del donut
ax_donut.text(0, 0, f"{len(df)}
PEPs", ha="center", va="center",
              fontsize=16, fontweight="bold", color=C_TEXT)

ax_donut.set_title("Distribución por Tipo de Organismo", pad=16)
ax_donut.set_facecolor(C_BG)

# Tabla de datos al costado
ax_table.axis("off")
table_data = [
    [org, str(cnt), f"{cnt/len(df)*100:.1f}%"]
    for org, cnt in org_counts.items()
]
table = ax_table.table(
    cellText=table_data,
    colLabels=["Tipo de Organismo", "PEPs", "%"],
    loc="center",
    cellLoc="left",
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1.1, 2.0)

# Estilo de tabla
for (row, col), cell in table.get_celld().items():
    cell.set_facecolor(C_BG2 if row > 0 else C_BG)
    cell.set_edgecolor("#2a2a3e")
    cell.set_text_props(color=C_TEXT if row > 0 else C_TEAL)
    if row > 0:
        cell.set_facecolor(DONUT_COLORS[row - 1] + "22" if row <= len(DONUT_COLORS) else C_BG2)

# Leyenda
legend_patches = [
    mpatches.Patch(color=DONUT_COLORS[i], label=org)
    for i, org in enumerate(org_counts.index)
]
ax_donut.legend(handles=legend_patches, loc="lower center",
               bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8,
               facecolor=C_BG, edgecolor=C_MUTED, framealpha=0.8)

fig.tight_layout()
plt.savefig("notebooks/viz_03_tipo_organismo.png", dpi=150,
            bbox_inches="tight", facecolor=C_BG)
plt.show()
print("✅ Gráfico guardado.")

### 4.4 PEPs en el Sector Financiero

El sector financiero es especialmente relevante en AML/KYC. Analizamos los **52 PEPs**
en instituciones financieras, diferenciando los 4 casos de **alto riesgo crítico**.

In [ ]:
fin_df = df[df["TIPO_ORGANISMO"] == "Instituciones Financieras"].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ---- Panel izquierdo: PEPs por institución financiera ----
ax1 = axes[0]
fin_inst = fin_df["ORGANISMO"].value_counts()

# Abreviar nombres
def abrev_fin(name):
    replacements = {
        "BANCO DE LA REPUBLICA ORIENTAL DEL URUGUAY (BROU)": "BROU",
        "BANCO DE SEGUROS DEL ESTADO (BSE)": "BSE",
        "BANCO DE PREVISION SOCIAL (BPS)": "BPS",
        "BANCO CENTRAL DEL URUGUAY (BCU)": "BCU",
        "BANCO HIPOTECARIO DEL URUGUAY (BHU)": "BHU",
    }
    for k, v in replacements.items():
        if k in name:
            return v
    return name

fin_inst.index = [abrev_fin(n) for n in fin_inst.index]

# Detectar instituciones con alto riesgo
alto_riesgo_inst = fin_df[fin_df["NIVEL_RIESGO"].str.contains("Alto")]["ORGANISMO"].apply(abrev_fin).unique()

bar_colors_fin = [C_RED if inst in alto_riesgo_inst else C_TEAL for inst in fin_inst.index]

bars = ax1.barh(fin_inst.index, fin_inst.values, color=bar_colors_fin, height=0.6, zorder=3)
for bar, val in zip(bars, fin_inst.values):
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             str(val), va="center", fontsize=10, color=C_TEXT)

ax1.set_title("PEPs por Institución Financiera")
ax1.set_xlabel("Cantidad de PEPs")
ax1.grid(axis="x", zorder=0)

legend_patches = [
    mpatches.Patch(color=C_RED, label="Con casos alto riesgo"),
    mpatches.Patch(color=C_TEAL, label="Riesgo medio"),
]
ax1.legend(handles=legend_patches, facecolor=C_BG, edgecolor=C_MUTED, fontsize=9)

# ---- Panel derecho: Casos críticos de alto riesgo ----
ax2 = axes[1]
ax2.axis("off")

criticos = fin_df[fin_df["NIVEL_RIESGO"].str.contains("Alto")][
    ["NOMBRE", "CARGO", "ORGANISMO"]
].copy()
criticos["ORGANISMO"] = criticos["ORGANISMO"].apply(abrev_fin)

# Formatear nombre
criticos["NOMBRE"] = criticos["NOMBRE"].apply(
    lambda n: " ".join(w.capitalize() for w in n.split())
)
criticos["CARGO"] = criticos["CARGO"].apply(
    lambda c: " ".join(w.capitalize() for w in c.split())
)

ax2.set_title("🚨 Casos Críticos: Alto Riesgo en Finanzas", color=C_RED, pad=14)

table_data = [[row["NOMBRE"], row["CARGO"], row["ORGANISMO"]] for _, row in criticos.iterrows()]
table = ax2.table(
    cellText=table_data,
    colLabels=["Nombre", "Cargo", "Institución"],
    loc="center",
    cellLoc="left",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2.5)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#2a2a3e")
    if row == 0:
        cell.set_facecolor(C_RED + "44")
        cell.set_text_props(color=C_RED, fontweight="bold")
    else:
        cell.set_facecolor(C_BG2)
        cell.set_text_props(color=C_TEXT)

fig.suptitle("Sector Financiero — 52 PEPs Registrados",
             fontsize=15, fontweight="bold", color=C_TEXT, y=1.02)
fig.tight_layout()
plt.savefig("notebooks/viz_04_sector_financiero.png", dpi=150,
            bbox_inches="tight", facecolor=C_BG)
plt.show()
print("✅ Gráfico guardado.")

### 4.5 Heatmap — Rango Etario por Tipo de Organismo

La edad estimada se infiere del número de CI uruguaya (mayor número → menor edad).
Este heatmap muestra la concentración demográfica en cada tipo de organismo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ---- Heatmap ----
ax1 = axes[0]

age_order   = ["Mayor 50 años", "35-50 años", "Menor 35 años"]
org_order   = df["TIPO_ORGANISMO"].value_counts().index.tolist()

heatmap_data = (
    df.groupby(["TIPO_ORGANISMO", "RANGO_EDAD"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=age_order, fill_value=0)
      .reindex(org_order)
)

cmap = sns.color_palette("YlOrRd", as_cmap=True)

sns.heatmap(
    heatmap_data,
    ax=ax1,
    cmap=cmap,
    annot=True,
    fmt="d",
    linewidths=0.5,
    linecolor="#0f0f17",
    cbar_kws={"label": "Cantidad de PEPs", "shrink": 0.85},
)

ax1.set_title("Concentración Etaria por Tipo de Organismo")
ax1.set_xlabel("Rango Etario Estimado", labelpad=10)
ax1.set_ylabel("")
ax1.tick_params(axis="x", rotation=0)

# Estilo del colorbar
cbar = ax1.collections[0].colorbar
cbar.ax.yaxis.label.set_color(C_TEXT)
cbar.ax.tick_params(colors=C_TEXT)

# ---- Barras apiladas: rango etario global ----
ax2 = axes[1]

age_counts = df["RANGO_EDAD"].value_counts().reindex(age_order)
AGE_COLORS = ["#457b9d", C_TEAL, "#a8dadc"]

bars = ax2.bar(age_counts.index, age_counts.values,
               color=AGE_COLORS, width=0.5, zorder=3)

for bar, val in zip(bars, age_counts.values):
    pct = val / len(df) * 100
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f"{val}
({pct:.1f}%)", ha="center", va="bottom", fontsize=10, color=C_TEXT)

ax2.set_title("Distribución Etaria Global")
ax2.set_ylabel("Cantidad de PEPs")
ax2.set_ylim(0, age_counts.max() * 1.2)
ax2.grid(axis="y", zorder=0)

fig.suptitle("Perfil Etario de los PEPs Uruguayos",
             fontsize=15, fontweight="bold", y=1.02)
fig.tight_layout()
plt.savefig("notebooks/viz_05_rango_etario.png", dpi=150,
            bbox_inches="tight", facecolor=C_BG)
plt.show()
print("✅ Gráfico guardado.")

## 5. 📝 Conclusiones

### Hallazgos Clave

1. **Alta concentración de riesgo**: El **40.1%** (344 de 857) de los PEPs pertenece a
   categorías de alto riesgo, lo que requiere diligencia reforzada permanente.

2. **Poder Judicial dominante**: Con 158 PEPs (18.4%), el Poder Judicial es el segundo
   tipo de organismo con más representación, y todos son de alto riesgo según la Ley 19.574.

3. **Sector financiero crítico**: Los 4 casos de alto riesgo en instituciones financieras
   (BSE, BROU, BHU) representan los perfiles de mayor prioridad para due diligence.

4. **Perfil etario predominante**: El 48.1% de los PEPs tiene entre 35 y 50 años,
   lo que sugiere una generación en pleno ejercicio de cargos de responsabilidad.

5. **Legislativo lidera en volumen**: El Poder Legislativo concentra el 28.5% de los PEPs,
   especialmente ediles departamentales (186 PEPs de representación local).

---

In [ ]:
# === TABLA RESUMEN EJECUTIVO ===

print("=" * 65)
print(" REPORTE EJECUTIVO — ANÁLISIS PEP URUGUAY (SENACLAFT)")
print("=" * 65)
print(f"  Total de PEPs registrados : {len(df):,}")
print(f"  Fuente                    : Lista oficial SENACLAFT")
print(f"  Ley aplicable             : N° 19.574 (AML/CFT)")
print()

print("DISTRIBUCIÓN POR NIVEL DE RIESGO:")
print("-" * 55)
for nivel, cnt in df["NIVEL_RIESGO"].value_counts().items():
    pct = cnt / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"  {nivel:<38} {cnt:>3}  {pct:>5.1f}%  {bar}")

print()
print("DISTRIBUCIÓN POR TIPO DE ORGANISMO:")
print("-" * 55)
for org, cnt in df["TIPO_ORGANISMO"].value_counts().items():
    pct = cnt / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"  {org:<38} {cnt:>3}  {pct:>5.1f}%  {bar}")

print()
print("CASOS CRÍTICOS — ALTO RIESGO EN SECTOR FINANCIERO:")
print("-" * 55)
criticos_full = df[
    df["NIVEL_RIESGO"].str.contains("Alto") &
    (df["TIPO_ORGANISMO"] == "Instituciones Financieras")
][["NOMBRE", "CARGO", "ORGANISMO"]]

for _, row in criticos_full.iterrows():
    nombre = " ".join(w.capitalize() for w in row["NOMBRE"].split())
    cargo  = " ".join(w.capitalize() for w in row["CARGO"].split())
    print(f"  • {nombre}")
    print(f"    {cargo} | {row['ORGANISMO']}")
    print()

print("=" * 65)

## 6. 💾 Exportación de Datos

Generamos los archivos CSV derivados para uso operativo en sistemas AML/KYC.

In [ ]:
# Exportar dataset enriquecido
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "pep_enriquecido.csv"), index=False)
print(f"✅ pep_enriquecido.csv          — {len(df):,} registros")

# Casos alto riesgo en finanzas
alto_fin = df[
    df["NIVEL_RIESGO"].str.contains("Alto") &
    (df["TIPO_ORGANISMO"] == "Instituciones Financieras")
][["CI", "NOMBRE", "CARGO", "ORGANISMO", "NIVEL_RIESGO"]]
alto_fin.to_csv(os.path.join(output_dir, "pep_alto_riesgo_finanzas.csv"), index=False)
print(f"✅ pep_alto_riesgo_finanzas.csv — {len(alto_fin)} registros")

# Riesgo medio en finanzas
medio_fin = df[
    ~df["NIVEL_RIESGO"].str.contains("Alto") &
    (df["TIPO_ORGANISMO"] == "Instituciones Financieras")
][["CI", "NOMBRE", "CARGO", "ORGANISMO", "NIVEL_RIESGO", "RANGO_EDAD"]]
medio_fin.to_csv(os.path.join(output_dir, "pep_medio_riesgo_finanzas.csv"), index=False)
print(f"✅ pep_medio_riesgo_finanzas.csv— {len(medio_fin)} registros")

print()
print("Archivos generados en el directorio data/")